# 02 - CNN training experiments

Reproduces the coursework experimental grid:

- 2 architectures: ResNet18, MobileNetV2
- 4 data scenarios: E1-E4
- 4 augmentation variants: A0-A3
- 32 runs in total

The fixed 20-image real evaluation set is identical for all runs.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install torch torchvision scikit-learn pandas openpyxl pillow

In [ ]:
import random, json
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

DATASET_ROOT = Path("/content/drive/MyDrive/dataset_cv")
TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"
OUT_DIR = DATASET_ROOT / "results_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 1337
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 12
LR = 3e-4
WEIGHT_DECAY = 1e-4
IMG_SIZE = 224

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
IMG_EXT = {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff",".jfif"}

def infer_domain(path: Path):
    parts = {p.lower() for p in path.parts}
    name = path.name.lower()
    if "real" in parts or "_real_" in name or name.startswith("real_") or name.endswith("_real.jpg"):
        return "real"
    if "synthetic" in parts or "synth" in parts or "_synthetic_" in name or "_synth_" in name:
        return "synthetic"
    return "unknown"

@dataclass
class Sample:
    path: Path
    label: int
    class_name: str
    domain: str

def build_index(split_dir: Path):
    assert split_dir.exists(), f"Missing folder: {split_dir}"
    class_names = sorted([p.name for p in split_dir.iterdir() if p.is_dir()])
    class_to_idx = {c:i for i,c in enumerate(class_names)}
    samples = []

    for c in class_names:
        files = [p for p in (split_dir/c).rglob("*")
                 if p.is_file() and p.suffix.lower() in IMG_EXT]
        for fp in files:
            samples.append(Sample(fp, class_to_idx[c], c, infer_domain(fp)))
    return samples, class_names, class_to_idx

train_samples, class_names, class_to_idx = build_index(TRAIN_DIR)
test_samples, test_class_names, _ = build_index(TEST_DIR)

assert class_names == test_class_names, "Train/test class names differ."
print("Classes:", class_names)
print("Train:", len(train_samples), Counter(s.domain for s in train_samples))
print("Test:", len(test_samples))

if len(train_samples) != 200:
    print("WARNING: coursework specification expects 200 training images.")
if len(test_samples) != 20:
    print("WARNING: coursework specification expects 20 test images.")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

AUGS = {
    "A0_none": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "A1_geom": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "A2_color": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "A3_mix": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}

class CVProjectDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s.path).convert("RGB")
        return self.transform(img), s.label

In [ ]:
def make_experiment_samples(all_train_samples, exp_name):
    if exp_name == "E1_all":
        return all_train_samples

    if exp_name == "E2_synth":
        return [s for s in all_train_samples if s.domain == "synthetic"]

    if exp_name == "E3_real":
        return [s for s in all_train_samples if s.domain == "real"]

    if exp_name == "E4_balanced_no_mix":
        real = [s for s in all_train_samples if s.domain == "real"]
        synth = [s for s in all_train_samples if s.domain == "synthetic"]

        if not real or not synth:
            raise ValueError("E4 requires both real and synthetic training images.")

        if len(synth) > len(real):
            majority, n, dom = synth, len(real), "synthetic"
        else:
            majority, n, dom = real, len(synth), "real"

        rng = random.Random(SEED)
        picked = rng.sample(majority, n)
        print(f"E4 -> domain={dom}, samples={n}")
        return picked

    raise ValueError(exp_name)

EXPERIMENTS = ["E1_all","E2_synth","E3_real","E4_balanced_no_mix"]

In [ ]:
def build_model(model_name, num_classes):
    if model_name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m

    if model_name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m

    raise ValueError(model_name)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    report = classification_report(
        y_true, y_pred,
        target_names=class_names,
        zero_division=0,
        output_dict=True
    )
    return acc, cm, report

In [ ]:
def train_one_run(model_name, exp_name, aug_name, epochs=EPOCHS):
    exp_samples = make_experiment_samples(train_samples, exp_name)
    if not exp_samples:
        raise ValueError(f"{exp_name} has no samples.")

    train_loader = DataLoader(
        CVProjectDataset(exp_samples, AUGS[aug_name]),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        CVProjectDataset(test_samples, test_tf),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = build_model(model_name, len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_acc = -1.0
    best_state = None
    best_cm = None
    best_report = None
    history = []

    for ep in range(1, epochs+1):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * x.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        acc, cm, report = evaluate(model, test_loader)

        history.append({"epoch": ep, "train_loss": train_loss, "test_acc": acc})

        if acc > best_acc:
            best_acc = acc
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            best_cm = cm
            best_report = report

        print(f"[{model_name} | {exp_name} | {aug_name}] "
              f"epoch {ep}/{epochs} loss={train_loss:.4f} test_acc={acc:.4f}")

    return best_acc, best_cm, best_report, history, best_state

In [ ]:
MODELS = ["resnet18","mobilenet_v2"]
AUG_NAMES = list(AUGS.keys())

results_rows = []

for model_name in MODELS:
    for exp_name in EXPERIMENTS:
        for aug_name in AUG_NAMES:
            run_id = f"{model_name}__{exp_name}__{aug_name}"
            run_dir = OUT_DIR / run_id
            run_dir.mkdir(parents=True, exist_ok=True)

            best_acc, best_cm, best_report, history, best_state = train_one_run(
                model_name, exp_name, aug_name
            )

            model_path = run_dir / "best_model.pth"
            torch.save(best_state, model_path)
            np.save(run_dir / "confusion_matrix.npy", best_cm)

            with open(run_dir / "classification_report.json", "w") as f:
                json.dump(best_report, f, indent=2)

            pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

            results_rows.append({
                "run_id": run_id,
                "model": model_name,
                "experiment": exp_name,
                "augmentation": aug_name,
                "best_test_acc": float(best_acc),
                "model_path": str(model_path),
            })

results_df = pd.DataFrame(results_rows).sort_values("best_test_acc", ascending=False)
results_df.to_csv(OUT_DIR / "results_summary.csv", index=False)
results_df.to_excel(OUT_DIR / "results_summary.xlsx", index=False)

display(results_df)
print("Saved to:", OUT_DIR)